1) Multiple tests. A patient tests positive twice on independent tests (both 99% accurate, disease prevalence 1 in 10,000). What is 
P(sick) after both tests? Use the posterior from the first test as the prior for the second.

In [ ]:

prevalence = 1 / 10000
accuracy = 0.99



sensitivity = 0.99 #Sensitivity = (TP/(TP+FN))
specificity = 0.99 # Specificity = (TN/(TN+FP))

# First test is positive
prior = prevalence

positive = sensitivity * prior + (1 - specificity) * (1 - prior)
posterior1 = (sensitivity * prior) / positive

# Second test is positive
prior = posterior1

positive = sensitivity * prior + (1 - specificity) * (1 - prior)
posterior2 = (sensitivity * prior) / positive

print("After first positive test:", posterior1)
print("After second positive test:", posterior2)
print("Probability of being sick:", posterior2 * 100, "%")

: 

2) Smoothing impact. Run the spam classifier with smoothing values of 0.01, 0.1, 1.0, and 10.0. How do the top word probabilities change? What happens with smoothing=0 and a word that appears only in ham?

In [1]:
# Word counts from training data
spam_word_count = 4
ham_word_count = 0

total_spam_words = 100
total_ham_words = 100
vocab_size = 10

smoothing_values = [0.01, 0.1, 1.0, 10.0]

for alpha in smoothing_values:
    spam_prob = (spam_word_count + alpha) / (
        total_spam_words + alpha * vocab_size
    )

    ham_prob = (ham_word_count + alpha) / (
        total_ham_words + alpha * vocab_size
    )

    print("Smoothing:", alpha)
    print("P(word | spam):", round(spam_prob, 4))
    print("P(word | ham):", round(ham_prob, 4))
    print()

Smoothing: 0.01
P(word | spam): 0.0401
P(word | ham): 0.0001

Smoothing: 0.1
P(word | spam): 0.0406
P(word | ham): 0.001

Smoothing: 1.0
P(word | spam): 0.0455
P(word | ham): 0.0091

Smoothing: 10.0
P(word | spam): 0.07
P(word | ham): 0.05



3)  Add features. Extend the NaiveBayes class to also use message length (short/long) as a feature alongside word counts. Estimate 
P
(
short
∣
spam
)
 and 
P
(
short
∣
ham
)
from the training data and fold it into the prediction score.

In [2]:
import math
from collections import Counter

# Training data
messages = [
    ("win free prize", "spam"),
    ("free money win", "spam"),
    ("claim prize now", "spam"),
    ("team meeting today", "ham"),
    ("project meeting tomorrow", "ham"),
    ("hello team today", "ham")
]

# Set the length threshold
def get_length(message):
    if len(message.split()) <= 3:
        return "short"
    return "long"

# Count words and message lengths for each class
spam_words = Counter()
ham_words = Counter()

spam_short = 0
ham_short = 0

spam_count = 0
ham_count = 0

for message, label in messages:
    words = message.split()

    if label == "spam":
        spam_count += 1
        spam_words.update(words)

        if get_length(message) == "short":
            spam_short += 1

    else:
        ham_count += 1
        ham_words.update(words)

        if get_length(message) == "short":
            ham_short += 1

# Vocabulary
vocab = set(spam_words) | set(ham_words)
V = len(vocab)

# Estimate length probabilities with Laplace smoothing
alpha = 1

p_short_spam = (spam_short + alpha) / (spam_count + 2 * alpha)
p_short_ham = (ham_short + alpha) / (ham_count + 2 * alpha)

print("P(short | spam):", p_short_spam)
print("P(short | ham):", p_short_ham)


# Predict a new message
def predict(message):
    words = message.split()
    length = get_length(message)

    # Start with log prior probabilities
    spam_score = math.log(spam_count / len(messages))
    ham_score = math.log(ham_count / len(messages))

    # Add word probabilities
    for word in words:
        spam_score += math.log(
            (spam_words[word] + alpha) /
            (sum(spam_words.values()) + alpha * V)
        )

        ham_score += math.log(
            (ham_words[word] + alpha) /
            (sum(ham_words.values()) + alpha * V)
        )

    # Add message-length probabilities
    if length == "short":
        spam_score += math.log(p_short_spam)
        ham_score += math.log(p_short_ham)
    else:
        spam_score += math.log(1 - p_short_spam)
        ham_score += math.log(1 - p_short_ham)

    if spam_score > ham_score:
        return "spam"
    return "ham"


print(predict("win free"))
print(predict("team meeting"))

P(short | spam): 0.8
P(short | ham): 0.8
spam
ham
